# Media Processing Results

> Standardized result DTOs for the media-processing task — the data nouns media-processing tool capabilities (ffmpeg today) emit and the multi-method task adapter / workflow cores consume, wire-registered so results cross the worker boundary typed.

**Canonical home (execution stage 8 — Option C / PILLAR 1c):** these DTOs live here in `cjm-capability-primitives` so a pure-compute media-processing tool depends only on the data nouns, never on the adapter machinery (`TaskAdapter`, the tool protocol, persistence). `media_processing` is a MULTI-METHOD task family (one tool, many ops), so it carries more than one result DTO: `MediaArtifactResult` (wire `"media_processing.artifact"`) for a single produced file (`convert`/`extract_audio`, the `SourceSeparationResult` shape); `MediaSegmentationResult` (wire `"media_processing.segmentation"`) for a batch of files (`segment_audio`, holding typed `MediaSegment`s + a custom `from_dict` per the `VADResult` precedent); and `MediaMetadata` (wire `"media_processing.metadata"`) for the uncached `get_info` probe (inline data, relocated from the dissolving `cjm-media-plugin-system.core`). See each class docstring for detail.

In [ ]:
#| default_exp media_processing

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from dataclasses import dataclass, field, asdict
from typing import Any, Dict, List

from cjm_substrate.core.wire import wire_type

In [ ]:
#| export
@dataclass
class MediaSegment:
    """One produced segment file from a `segment_audio` batch cut.

    The per-segment entry the fused-era ffmpeg `segment_audio` returned as a
    dict, now a typed noun (the dead `job_id` dropped — born-final; the adapter
    owns persistence). Workflow cores read `index`/`output_path`/`start`/`end`
    to build the per-segment composition."""
    index: int        # 0-based position of this segment within the batch
    output_path: str  # Path to the produced segment file the tool wrote
    start: float      # Segment start time in the source (seconds)
    end: float        # Segment end time in the source (seconds)
    duration: float   # end - start (seconds)

    def to_dict(self) -> Dict[str, Any]:  # Serialized representation
        """Convert to dictionary for JSON serialization."""
        return asdict(self)

In [ ]:
#| export
@wire_type("media_processing.artifact")
@dataclass
class MediaArtifactResult:
    """A single produced audio artifact (the `convert` / `extract_audio` output).

    The artifact-producing shape (cf. `SourceSeparationResult`): `output_path`
    is the file the tool wrote to the adapter-chosen location; `metadata`
    carries the stats the fused-era return dict / row held (codec, duration,
    stream_copy, the effective convert parameters, ...). Flat fields (str +
    dict), so the default wire reconstruction suffices — no custom from_dict."""
    output_path: str                                        # Path to the produced audio file
    metadata: Dict[str, Any] = field(default_factory=dict)  # Stats (codec, duration, parameters, ...)

In [ ]:
#| export
@wire_type("media_processing.segmentation")
@dataclass
class MediaSegmentationResult:
    """A BATCH of produced segment files (the `segment_audio` output).

    Holds typed `MediaSegment`s plus the batch metadata the fused-era return
    dict carried (`input_path`, `segment_count`, `total_duration`, `batch_key`
    — the label linking the cut files in the run manifest). Because `segments`
    holds typed objects, a custom `from_dict` re-types them on wire-decode (the
    auto flat reconstruct would leave bare dicts, breaking `seg.output_path`
    access) — the `VADResult` precedent."""
    segments: List[MediaSegment]          # The produced segment files, ordered by index
    input_path: str = ""                  # The source audio that was cut
    segment_count: int = 0                # Number of segments produced
    total_duration: float = 0.0           # Sum of segment durations (seconds)
    batch_key: str = ""                   # Label linking this batch's cut files (run-manifest field)

    @classmethod
    def from_dict(
        cls,
        d: Dict[str, Any],  # Wire payload (the substrate envelope's "data")
    ) -> "MediaSegmentationResult":  # Reconstructed result with typed segments
        """Reconstruct from a wire payload, re-typing nested MediaSegments."""
        return cls(
            segments=[s if isinstance(s, MediaSegment) else MediaSegment(**s)
                      for s in (d.get("segments") or [])],
            input_path=d.get("input_path") or "",
            segment_count=int(d.get("segment_count") or 0),
            total_duration=float(d.get("total_duration") or 0.0),
            batch_key=d.get("batch_key") or "",
        )

In [ ]:
#| export
@wire_type("media_processing.metadata")
@dataclass
class MediaMetadata:
    """Probed metadata for a media file (the `get_info` result) — inline data, no artifact.

    Relocated to `cjm-capability-primitives` from the dissolving
    `cjm-media-plugin-system.core` (the `TranscriptionResult`/`ForcedAlignResult`
    relocation precedent). `get_info` is the media-processing task's UNCACHED
    probe op, so this is a read result, not a produced-artifact pointer. The
    stream lists are plain dicts, so the default wire reconstruction suffices."""
    path: str                                                          # File path probed
    duration: float                                                    # Duration in seconds
    format: str                                                        # Container format (e.g. 'mp4', 'mkv')
    size_bytes: int                                                    # File size in bytes
    video_streams: List[Dict[str, Any]] = field(default_factory=list)  # Per-video-stream info (codec, width, height, fps)
    audio_streams: List[Dict[str, Any]] = field(default_factory=list)  # Per-audio-stream info (codec, sample_rate, channels, duration)

    def to_dict(self) -> Dict[str, Any]:  # Serialized representation
        """Convert to dictionary for JSON serialization."""
        return asdict(self)

In [ ]:
# Test MediaArtifactResult (the convert / extract_audio output)
result = MediaArtifactResult(
    output_path="/data/cjm-media-plugin-ffmpeg/convert/ab12cd_0f1e2d3c4b5a/episode.wav",
    metadata={"output_format": "wav", "sample_rate": 16000, "channels": 1,
              "resampler": "soxr", "duration": 300.0},
)
print(f"Output path: {result.output_path}")
print(f"Metadata: {result.metadata}")
assert result.output_path.endswith("episode.wav")
assert result.metadata["sample_rate"] == 16000

# Minimal (just the produced path; empty metadata)
minimal = MediaArtifactResult(output_path="/tmp/out.wav")
assert minimal.output_path == "/tmp/out.wav" and minimal.metadata == {}

In [ ]:
# Test MediaSegmentationResult (the segment_audio batch output) + MediaMetadata
seg_result = MediaSegmentationResult(
    segments=[
        MediaSegment(index=0, output_path="/tmp/seg/segment_000.wav", start=0.0, end=2.5, duration=2.5),
        MediaSegment(index=1, output_path="/tmp/seg/segment_001.wav", start=2.5, end=6.0, duration=3.5),
    ],
    input_path="/data/episode.mp3", segment_count=2, total_duration=6.0, batch_key="batch-abc",
)
print(f"Segments: {seg_result.segments}")
assert seg_result.segments[0].index == 0 and seg_result.segments[1].output_path.endswith("segment_001.wav")
assert seg_result.segment_count == 2 and seg_result.batch_key == "batch-abc"

# Empty batch (defaults)
empty = MediaSegmentationResult(segments=[])
assert empty.segments == [] and empty.segment_count == 0 and empty.batch_key == ""

# MediaMetadata (the get_info probe result)
meta = MediaMetadata(
    path="/data/episode.mp3", duration=300.0, format="mp3", size_bytes=7200000,
    audio_streams=[{"codec": "mp3", "sample_rate": 44100, "channels": 2, "duration": 300.0}],
)
print(f"Metadata: {meta}")
assert meta.duration == 300.0 and meta.audio_streams[0]["sample_rate"] == 44100 and meta.video_streams == []

In [ ]:
# Wire-format executable test: each result round-trips TYPED through the
# simulated worker boundary (encode -> json -> decode). MediaSegmentationResult
# carries a custom from_dict re-typing its nested MediaSegments (the auto flat
# reconstruct would leave bare dicts); MediaArtifactResult / MediaMetadata are
# flat and use the default reconstruction.
import json as _json
from cjm_substrate.core.wire import wire_encode, wire_decode, WIRE_KIND_KEY

# media_processing.artifact
art = MediaArtifactResult(output_path="/data/convert/out.wav",
                          metadata={"output_format": "wav", "sample_rate": 16000})
env = wire_encode(art)
assert env[WIRE_KIND_KEY] == "media_processing.artifact"
back = wire_decode(_json.loads(_json.dumps(env)))
assert isinstance(back, MediaArtifactResult) and back == art

# media_processing.segmentation (nested MediaSegments must come back typed)
seg = MediaSegmentationResult(
    segments=[MediaSegment(index=0, output_path="/tmp/s0.wav", start=0.0, end=2.5, duration=2.5)],
    input_path="/data/episode.mp3", segment_count=1, total_duration=2.5, batch_key="b1",
)
env = wire_encode(seg)
assert env[WIRE_KIND_KEY] == "media_processing.segmentation"
back = wire_decode(_json.loads(_json.dumps(env)))
assert isinstance(back, MediaSegmentationResult)
assert all(isinstance(s, MediaSegment) for s in back.segments)
assert back == seg

# media_processing.metadata
meta = MediaMetadata(path="/data/episode.mp3", duration=300.0, format="mp3", size_bytes=10)
env = wire_encode(meta)
assert env[WIRE_KIND_KEY] == "media_processing.metadata"
back = wire_decode(_json.loads(_json.dumps(env)))
assert isinstance(back, MediaMetadata) and back == meta

print("media_processing wire round-trips OK (artifact / segmentation / metadata)")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()